# Silver project source match

## Purpose

Reconcile usable flood-control Contract IDs to canonical DPWH projects using exact normalized Contract ID equality only. The output records match evidence and never mutates `silver_project`, assigns geography, or combines source budgets.


## 1. Confirm upstream table safety

Both upstream Silver tables must be nonempty, unique at their documented grains, and linked to exactly one selected run.


In [0]:
%sql
USE CATALOG `buildabida-capstone`;

SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM `02-silver`.silver_flood_control_component) > 0
    AND (SELECT COUNT(*) FROM `02-silver`.silver_project) > 0,
    'STOP: source reconciliation requires nonempty component and project tables'
);

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT flood_component_key)
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT run_id) = 1,
    'STOP: flood-control component grain or run selection is unsafe'
)
FROM `02-silver`.silver_flood_control_component;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT project_key)
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT run_id) = 1,
    'STOP: canonical project grain or run selection is unsafe'
)
FROM `02-silver`.silver_project;


## 2. Group source components without losing evidence

The match table contains one usable normalized source Contract ID per selected flood snapshot, source version, and match run. Invalid IDs remain in the component table and are counted by validation. Repeated source components are summarized explicitly; repeated cost values are never blindly summed.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW flood_source_contract_groups AS
SELECT
    source_contract_id_normalized,
    source_snapshot_id,
    source_version,
    run_id AS source_component_run_id,
    transformation_rule_version AS source_component_rule_version,
    COUNT(*) AS source_component_count,
    COUNT(DISTINCT source_contract_id_raw) AS source_raw_contract_id_count,
    SORT_ARRAY(COLLECT_SET(source_contract_id_raw)) AS source_raw_contract_ids,
    COUNT_IF(source_contract_cost_status = 'VALID') AS valid_source_cost_count,
    COUNT(DISTINCT CASE
        WHEN source_contract_cost_status = 'VALID'
            THEN source_contract_cost_pesos
    END) AS distinct_valid_source_cost_count,
    CASE
        WHEN COUNT(DISTINCT CASE
            WHEN source_contract_cost_status = 'VALID'
                THEN source_contract_cost_pesos
        END) = 1
            THEN MAX(CASE
                WHEN source_contract_cost_status = 'VALID'
                    THEN source_contract_cost_pesos
            END)
        ELSE NULL
    END AS resolved_source_contract_cost_pesos,
    CASE
        WHEN COUNT(DISTINCT CASE
            WHEN source_contract_cost_status = 'VALID'
                THEN source_contract_cost_pesos
        END) = 0 THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT CASE
            WHEN source_contract_cost_status = 'VALID'
                THEN source_contract_cost_pesos
        END) = 1 THEN 'RESOLVED_IDENTICAL'
        ELSE 'CONFLICT'
    END AS source_cost_resolution_status,
    MIN(source_load_ts) AS source_load_ts
FROM `02-silver`.silver_flood_control_component
WHERE source_contract_id_normalized IS NOT NULL
GROUP BY
    source_contract_id_normalized,
    source_snapshot_id,
    source_version,
    run_id,
    transformation_rule_version;

CREATE OR REPLACE TEMPORARY VIEW project_contract_candidates AS
SELECT
    UPPER(TRIM(contract_id)) AS target_contract_id_normalized,
    COUNT(*) AS target_candidate_count,
    CASE WHEN COUNT(*) = 1 THEN MAX(project_key) END AS matched_project_key,
    CASE WHEN COUNT(*) = 1 THEN MAX(contract_id) END AS matched_project_contract_id,
    CASE WHEN COUNT(*) = 1 THEN MAX(run_id) END AS matched_project_run_id,
    CASE WHEN COUNT(*) = 1 THEN MAX(source_snapshot_id) END
        AS matched_project_source_snapshot_id,
    CASE WHEN COUNT(*) = 1 THEN MAX(source_ingest_run_id) END
        AS matched_project_source_ingest_run_id,
    CASE WHEN COUNT(*) = 1 THEN MAX(transformation_rule_version) END
        AS matched_project_rule_version
FROM `02-silver`.silver_project
WHERE contract_id IS NOT NULL
  AND TRIM(contract_id) <> ''
GROUP BY UPPER(TRIM(contract_id));


## 3. Publish exact-match evidence

The status is `MATCHED_EXACT_CONTRACT_ID`, `UNMATCHED`, or `AMBIGUOUS_TARGET`. A target is published only when exactly one candidate exists. No arbitrary candidate selection is allowed.


In [0]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_project_source_match AS
WITH match_input AS (
    SELECT
        source.*,
        COALESCE(target.target_candidate_count, 0) AS target_candidate_count,
        target.matched_project_key,
        target.matched_project_contract_id,
        target.matched_project_run_id,
        target.matched_project_source_snapshot_id,
        target.matched_project_source_ingest_run_id,
        target.matched_project_rule_version,
        project_run.project_run_id AS target_project_run_id
    FROM flood_source_contract_groups AS source
    LEFT JOIN project_contract_candidates AS target
        ON source.source_contract_id_normalized
        = target.target_contract_id_normalized
    CROSS JOIN (
        SELECT MAX(run_id) AS project_run_id
        FROM `02-silver`.silver_project
    ) AS project_run
)
SELECT
    SHA2(
        CONCAT_WS(
            '||',
            source_contract_id_normalized,
            COALESCE(source_snapshot_id, ''),
            COALESCE(source_version, ''),
            COALESCE(source_component_run_id, ''),
            COALESCE(target_project_run_id, ''),
            'silver_project_source_match_v1'
        ),
        256
    ) AS project_source_match_key,
    source_contract_id_normalized,
    source_raw_contract_id_count,
    source_raw_contract_ids,
    source_component_count,
    valid_source_cost_count,
    distinct_valid_source_cost_count,
    resolved_source_contract_cost_pesos,
    source_cost_resolution_status,
    target_candidate_count,
    CASE
        WHEN target_candidate_count = 1 THEN 'MATCHED_EXACT_CONTRACT_ID'
        WHEN target_candidate_count = 0 THEN 'UNMATCHED'
        ELSE 'AMBIGUOUS_TARGET'
    END AS match_status,
    'EXACT_CONTRACT_ID' AS match_method,
    CASE WHEN target_candidate_count = 1 THEN 'EXACT' ELSE NULL END AS match_quality,
    CASE WHEN target_candidate_count = 1 THEN matched_project_key END
        AS matched_project_key,
    CASE WHEN target_candidate_count = 1 THEN matched_project_contract_id END
        AS matched_project_contract_id,
    CASE WHEN target_candidate_count = 1 THEN matched_project_run_id END
        AS matched_project_run_id,
    CASE WHEN target_candidate_count = 1 THEN matched_project_source_snapshot_id END
        AS matched_project_source_snapshot_id,
    CASE WHEN target_candidate_count = 1 THEN matched_project_source_ingest_run_id END
        AS matched_project_source_ingest_run_id,
    CASE WHEN target_candidate_count = 1 THEN matched_project_rule_version END
        AS matched_project_rule_version,
    source_snapshot_id,
    source_version,
    source_component_run_id,
    source_component_rule_version,
    target_project_run_id,
    source_load_ts,
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(source_snapshot_id, ''),
            COALESCE(source_component_run_id, ''),
            COALESCE(target_project_run_id, ''),
            'silver_project_source_match_v1'
        ),
        256
    ) AS run_id,
    'silver_project_source_match_v1' AS transformation_rule_version
FROM match_input;


# Summary

Created `02-silver`.silver_project_source_match at one usable normalized flood-source Contract ID per selected source snapshot, source version, and deterministic match run. Exact normalized Contract ID is the only method. Invalid source IDs remain accounted for in the component table and validation. D-04 remains open because this evidence does not authorize source consolidation or budget addition.

Requires Databricks execution.
